# Explorer 05 - Snapshot comparison

**Purpose.** Diff January 2020 against October 2026: which labels changed parent, and how late releases get catalogued.

**Feeds main report section:** 5.4 Snapshot comparison, and 7. Limitations

**Modules used:** `vinyl.snapshots`, `vinyl.labels`, `vinyl.viz`

**Tables read:** `labels` and `releases` from both snapshots

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-05-snapshots", shuffle_partitions=8)
from vinyl import snapshots, viz

In [ ]:
# Cache the labels tables: the root walk and the parent chains read them many times.
old, new = config.BASE_SNAPSHOT, config.MAIN_SNAPSHOT   # "2020", "2026"
label_cols = ["label_id", "label_name", "parent_id", "parent_name"]
labels_old = ingest.read_table(spark, old, "labels").select(*label_cols).cache()
labels_new = ingest.read_table(spark, new, "labels").select(*label_cols).cache()
releases_old, releases_new = ingest.read_table(spark, old, "releases"), ingest.read_table(spark, new, "releases")

## 1. Labels whose parent changed

In [ ]:
changes = snapshots.label_parent_changes(labels_old, labels_new).cache()
changes.groupBy("change").count().orderBy(F.desc("count")).show()

In [ ]:
# Biggest ownership changes: rank changed labels by how many in-scope releases they have.
release_labels = ingest.read_table(spark, new, "release_labels")
volume = release_labels.groupBy("label_id").agg(F.countDistinct("release_id").alias("n_releases"))
(changes.where(F.col("change").isin("changed_parent", "gained_parent", "lost_parent"))
    .join(volume, "label_id").orderBy(F.desc("n_releases")).show(30, truncate=False))

### Do the three majors' root labels exist in both snapshots?

`config.MAJOR_ROOT_IDS` was checked against the 2026 table only. A root that is missing in 2020, or has a different parent there, is printed below.

In [ ]:
roots = snapshots.major_root_check(labels_old, labels_new)
print(roots.to_string(index=False))
for r in roots[~roots.ok].itertuples():
    print(f"CHECK {r.major} (root {r.root_id}): in 2020={r.in_old}, in 2026={r.in_new}, "
          f"parent in 2020={r.parent_id_old}, parent in 2026={r.parent_id_new}")
if roots.ok.all():
    print("All three roots exist in both snapshots with the same parent.")

### Moves into and out of a major tree

Roots are resolved in each snapshot separately (placeholder parent 212 is not followed). `n_releases` counts distinct releases on the label in the 2026 `release_labels` table.

In [ ]:
moves = snapshots.major_moves(labels_old, labels_new, release_labels).cache()
print(f"{moves.count():,} labels changed major between the snapshots")
(moves.groupBy("major_old", "major_new")
    .agg(F.count("*").alias("n_labels"), F.sum("n_releases").alias("n_releases"))
    .orderBy(F.desc("n_labels")).show(truncate=False))

In [ ]:
(moves.select("label_id", "label_name", "major_old", "major_new", "move", "n_releases")
    .show(30, truncate=False))

### Hand-check the top moves

**Reading rule.** A real acquisition is a new parent that is a known corporate entity, often with an `(n)` suffix in its name, for example a Warner company. A data correction is a parent that flips between a placeholder, a duplicate or a similar-name profile, or a fix to the chain above the label.

Each table below is the parent chain of one label in the 2020 (`old`) and the 2026 (`new`) snapshot.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
for row in moves.limit(5).collect():
    print(f"=== {row.label_id} {row.label_name}: {row.major_old} -> {row.major_new} ({row.n_releases:,} releases)")
    print(snapshots.explain_move(row.label_id, labels_old, labels_new).drop(columns="label_id").to_string(index=False))

## 2. Cataloguing lag

In [ ]:
lag = snapshots.cataloguing_lag(releases_old, releases_new)
lag_pd = lag.toPandas()
lag_pd

In [ ]:
viz.plot_lines(lag_pd[lag_pd.year <= 2019], "year", "completeness",
               title="Share of each year's releases already catalogued by January 2020, release years 2016-2019",
               ylabel="share already catalogued", percent=True)

Completeness by age, where `age = 2020 - year`. Only `year <= 2019` is used: 2020 in the January 2020 snapshot is a stub of a few days.

In [ ]:
by_age = lag_pd[lag_pd.year <= 2019].assign(age=lambda d: int(old) - d.year)
print(by_age[["age", "year", "n_new", "n_already", "n_added_later", "completeness"]]
      .sort_values("age").to_string(index=False))

## 3. Estimated undercount of recent years

In [ ]:
yearly = releases_new.groupBy("year").agg(F.count("*").alias("n_releases"))
est_pd = snapshots.estimate_undercount(lag, yearly).toPandas()
est_pd[["year", "n_releases", "age", "completeness", "partial_year", "n_estimated"]]

In [ ]:
long = est_pd.melt(id_vars="year", value_vars=["n_releases", "n_estimated"], var_name="series", value_name="n")
ax = viz.plot_lines(long, "year", "n", series="series", ylabel="releases",
                    title="Releases per year as catalogued (n_releases) and after the undercount estimate (n_estimated)")
for y in est_pd.loc[est_pd.partial_year, "year"]:
    ax.axvspan(y - 0.5, y + 0.5, color="grey", alpha=0.2)
    ax.text(y, ax.get_ylim()[1], f"{y}\npartial year", ha="center", va="top", fontsize=8)

The same estimate without the seasonal correction (`partial_year_share={}`), next to the corrected one, to show the size of the correction for the partial year.

In [ ]:
est_flat = snapshots.estimate_undercount(lag, yearly, partial_year_share={}).toPandas()
compare = est_pd[["year", "n_releases", "partial_year", "n_estimated"]].merge(
    est_flat[["year", "n_estimated"]].rename(columns={"n_estimated": "n_estimated_no_seasonal"}), on="year")
compare["seasonal_factor"] = compare.n_estimated / compare.n_estimated_no_seasonal
print(compare.to_string(index=False))

## 4. Does the lag differ by format or genre?

Only the 2026 tables carry formats and genres, so each release is grouped by its 2026 formats or genres. A release with two formats counts once in each. `year <= 2019` only, as above.

In [ ]:
early = releases_new.where(F.col("year") <= 2019).select("release_id", "year").cache()
formats_new = ingest.read_table(spark, new, "release_formats")
lag_fmt = snapshots.cataloguing_lag_by(
    releases_old, early, formats_new.where(F.col("format_name").isin("Vinyl", "CD", "Cassette", "File")),
    "format_name").toPandas().assign(age=lambda d: int(old) - d.year)
print(lag_fmt.pivot(index=["age", "year"], columns="format_name", values="completeness").round(3).to_string())
print(lag_fmt.pivot(index=["age", "year"], columns="format_name", values="n_new").to_string())
viz.plot_lines(lag_fmt, "year", "completeness", series="format_name", percent=True,
               title="Share already catalogued by January 2020, by format, release years 2016-2019",
               ylabel="share already catalogued")

In [ ]:
genres_new = ingest.read_table(spark, new, "release_genres")
top_genres = [r.genre for r in (genres_new.where(F.col("year") <= 2019).groupBy("genre")
              .agg(F.countDistinct("release_id").alias("n")).orderBy(F.desc("n")).limit(8).collect())]
print(top_genres)
lag_gen = snapshots.cataloguing_lag_by(
    releases_old, early, genres_new.where(F.col("genre").isin(top_genres)), "genre"
).toPandas().assign(age=lambda d: int(old) - d.year)
print(lag_gen.pivot(index=["age", "year"], columns="genre", values="completeness").round(3).to_string())
print(lag_gen.pivot(index=["age", "year"], columns="genre", values="n_new").to_string())
viz.plot_lines(lag_gen, "year", "completeness", series="genre", percent=True,
               title="Share already catalogued by January 2020, by genre (top 8), release years 2016-2019",
               ylabel="share already catalogued")

**Reading rule.** If completeness for the same age differs between groups, then in the October 2026 snapshot the most recent years are undercounted by different amounts per group, and recent-year shares are biased toward the faster-catalogued groups. If the groups have about the same completeness at each age, shares are not biased by the lag, even though counts are.

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] List of confirmed ownership changes worth naming in the report.
- [ ] Completeness by age (the correction table) and which years are flagged as undercounted.
- [ ] Whether trend charts stop at 2024 or show 2025-2026 with a caveat.